In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds

In [3]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=['train','test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)



In [7]:
# TextVectorization layer for tokenization and vocabulary creation
max_tokens = 20000  # Maximum number of unique tokens
max_len = 100       # Maximum sequence length

vectorizer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_len
)

# Adapt vectorizer on the training dataset (text only)
train_texts = ds_train.map(lambda text, label: text)
vectorizer.adapt(train_texts)

# Function to encode the dataset
def encode_map(text, label):
    encoded_text = vectorizer(text)
    return encoded_text, label

AUTOTUNE = tf.data.AUTOTUNE
ds_train = ds_train.map(encode_map, num_parallel_calls=AUTOTUNE).cache()
ds_train = ds_train.shuffle(10000)
ds_train = ds_train.batch(32).prefetch(AUTOTUNE)

ds_test = ds_test.map(encode_map)
ds_test = ds_test.batch(32)

# Model definition
model = keras.Sequential([
    layers.Embedding(input_dim=max_tokens, output_dim=32, mask_zero=True),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)  # Binary classification
])

model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(3e-4, clipnorm=1),
    metrics=['accuracy']
)

# Train and evaluate the model
model.fit(ds_train, epochs=10, verbose=2)
loss, accuracy = model.evaluate(ds_test, verbose=2)
print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")


Epoch 1/10
782/782 - 8s - 11ms/step - accuracy: 0.6259 - loss: 0.5854
Epoch 2/10
782/782 - 6s - 7ms/step - accuracy: 0.8336 - loss: 0.3648
Epoch 3/10
782/782 - 6s - 7ms/step - accuracy: 0.8771 - loss: 0.2863
Epoch 4/10
782/782 - 6s - 7ms/step - accuracy: 0.9014 - loss: 0.2376
Epoch 5/10
782/782 - 6s - 7ms/step - accuracy: 0.9208 - loss: 0.1999
Epoch 6/10
782/782 - 6s - 7ms/step - accuracy: 0.9339 - loss: 0.1692
Epoch 7/10
782/782 - 6s - 7ms/step - accuracy: 0.9464 - loss: 0.1426
Epoch 8/10
782/782 - 6s - 7ms/step - accuracy: 0.9582 - loss: 0.1189
Epoch 9/10
782/782 - 6s - 7ms/step - accuracy: 0.9664 - loss: 0.0986
Epoch 10/10
782/782 - 6s - 7ms/step - accuracy: 0.9737 - loss: 0.0810
782/782 - 6s - 8ms/step - accuracy: 0.7858 - loss: 0.6574
Test Loss: 0.6574074029922485, Test Accuracy: 0.7858399748802185
